# Detection Retrain v2 — Phase Training + Crash Recovery

Fresh detection training with phase approach (like segmentation):
- **Phase 1:** 320px, 20 epochs, frozen backbone
- **Phase 2:** 640px, 50 epochs, unfrozen
- **Phase 3:** 1024px, 30 epochs, fine-tune (if time/GPU)

Key techniques:
- OmniCrack-only (skip DACL10K multi-class noise that broke notebook 02)
- Phase training at INCREASING resolutions (not stuck at 640px)
- Strong augmentation (geometry + intensity)
- Confidence threshold sweep (0.25 → 0.7)
- **history.json crash recovery** (resume from checkpoint)

**Goal: mAP@50 > 0.60 (heritage domain)**

## Setup

In [ ]:
import subprocess, sys
for pkg in ['ultralytics', 'torch', 'torchvision', 'opencv-python', 'scikit-learn', 'pyyaml']:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    except:
        pass
print('Ready')

In [ ]:
import os, json, zipfile, shutil, random
from pathlib import Path
from datetime import datetime
import cv2, numpy as np, yaml, torch
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR = Path('/content/drive/MyDrive/HeritagePreservation')
    DATA_DIR = Path('/content/data')
    CHECKPOINT_DIR = Path('/content/checkpoints/detector_v2_phase')
except:
    IN_COLAB = False
    DATA_DIR = Path.cwd().parent / 'data'
    CHECKPOINT_DIR = Path.cwd().parent / 'detection_checkpoint_v2_phase'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data: {DATA_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

## Extract & Load Data

In [ ]:
def extract_dataset(zip_name, data_dir, drive_dir):
    zip_path = drive_dir / zip_name
    out_dir = data_dir / zip_name.replace('.zip', '')
    if not zip_path.exists() or (out_dir.exists() and any(out_dir.rglob('*.*'))):
        return
    size_mb = zip_path.stat().st_size / 1e6
    local_zip = Path(f'/content/_tmp.zip')
    print(f'Copying {size_mb:.0f} MB...')
    shutil.copy2(zip_path, local_zip)
    print(f'Extracting...')
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(data_dir)
    local_zip.unlink()

if IN_COLAB:
    extract_dataset('omnicrack30k.zip', DATA_DIR, DRIVE_DIR)

# Find OmniCrack
OMNI = DATA_DIR / 'omnicrack30k' if (DATA_DIR / 'omnicrack30k').exists() else DATA_DIR
IMG_ROOT = OMNI / 'images'
ANN_ROOT = OMNI / 'annotations'
SPLITS = ['training', 'validation', 'test']
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

# Build mask lookup
masks = {}
for split in SPLITS:
    for m in (ANN_ROOT / split).rglob('*.*') if (ANN_ROOT / split).exists() else []:
        if m.suffix.lower() in IMG_EXTS:
            masks[m.stem] = m

# Find pairs
pairs = []
for split in SPLITS:
    for img in (IMG_ROOT / split).rglob('*.*') if (IMG_ROOT / split).exists() else []:
        if img.suffix.lower() in IMG_EXTS and img.stem in masks:
            pairs.append((str(img), str(masks[img.stem])))

print(f'Found: {len(pairs)} pairs')
if len(pairs) > 5000:
    random.Random(42).shuffle(pairs)
    pairs = pairs[:5000]

train, temp = train_test_split(pairs, test_size=0.30, random_state=42)
val, test = train_test_split(temp, test_size=0.50, random_state=42)
print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

## Convert Masks to YOLO

In [ ]:
def mask_to_yolo(mask_path):
    m = cv2.imread(str(mask_path), 0)
    if m is None:
        return []
    h, w = m.shape
    _, b = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    labels = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < 200:
            continue
        x, y, bw, bh = cv2.boundingRect(c)
        bwn, bhn = bw/w, bh/h
        if bwn*bhn > 0.40 or max(bwn/(bhn+1e-6), bhn/(bwn+1e-6)) > 8.0:
            continue
        cx, cy = (x+bw/2)/w, (y+bh/2)/h
        labels.append(f'0 {cx:.6f} {cy:.6f} {min(bwn,1):.6f} {min(bhn,1):.6f}')
    return labels

lbl_dir = OMNI / 'labels'
lbl_dir.mkdir(parents=True, exist_ok=True)

print('Converting...')
valid = []
for img_path, mask_path in tqdm(pairs):
    labels = mask_to_yolo(mask_path)
    if labels:
        rel = Path(img_path).relative_to(IMG_ROOT)
        lbl = lbl_dir / rel.with_suffix('.txt')
        lbl.parent.mkdir(parents=True, exist_ok=True)
        lbl.write_text('\n'.join(labels))
        valid.append((img_path, str(lbl)))

print(f'Valid: {len(valid)}')
train, temp = train_test_split(valid, test_size=0.30, random_state=42)
val, test = train_test_split(temp, test_size=0.50, random_state=42)
pairs = valid

## Setup Dataset

In [ ]:
for d in ['train', 'val', 'test']:
    for sub in ['images', 'labels']:
        (CHECKPOINT_DIR / d / sub).mkdir(parents=True, exist_ok=True)

def link_split(samples, split):
    for img, lbl in samples:
        try:
            os.symlink(img, CHECKPOINT_DIR / split / 'images' / Path(img).name)
        except:
            shutil.copy2(img, CHECKPOINT_DIR / split / 'images' / Path(img).name)
        shutil.copy2(lbl, CHECKPOINT_DIR / split / 'labels' / Path(lbl).name)

link_split(train, 'train')
link_split(val, 'val')
link_split(test, 'test')

yaml_path = CHECKPOINT_DIR / 'dataset.yaml'
yaml.dump({
    'path': str(CHECKPOINT_DIR),
    'train': 'train',
    'val': 'val',
    'test': 'test',
    'nc': 1,
    'names': {0: 'crack'}
}, open(yaml_path, 'w'))

print(f'Train: {len(list((CHECKPOINT_DIR/"train"/"images").glob("*")))} images')

## History & Recovery

In [ ]:
hist_file = CHECKPOINT_DIR / 'history.json'
hist = {
    'p1_done': False, 'p2_done': False, 'p3_done': False,
    'p1_map': 0, 'p2_map': 0, 'p3_map': 0,
}

if hist_file.exists():
    hist = json.load(open(hist_file))
    print(f'Phase 1: {hist["p1_done"]} ({hist["p1_map"]:.4f})')
    print(f'Phase 2: {hist["p2_done"]} ({hist["p2_map"]:.4f})')
    print(f'Phase 3: {hist["p3_done"]} ({hist["p3_map"]:.4f})')
else:
    print('Fresh start')

def save_hist():
    json.dump(hist, open(hist_file, 'w'), indent=2)

## Phase 1: 320px

In [ ]:
if not hist['p1_done']:
    print('\n=== PHASE 1: 320px (20 epochs) ===')
    m = YOLO('yolov8l.pt')
    r = m.train(
        data=str(yaml_path), imgsz=320, epochs=20, batch=16,
        device=0 if torch.cuda.is_available() else -1,
        patience=10, save=True, save_period=5,
        project=str(CHECKPOINT_DIR), name='phase1_320px', exist_ok=True,
        lr0=0.001, warmup_epochs=10, warmup_momentum=0.8, momentum=0.937, weight_decay=0.0005,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=15, translate=0.1, scale=0.5,
        flipud=0.5, fliplr=0.5, mosaic=1.0, val=True, verbose=True
    )
    best = Path(r.save_dir) / 'weights' / 'best.pt'
    if best.exists():
        v = YOLO(str(best)).val(data=str(yaml_path), split='val', verbose=False)
        hist['p1_map'] = float(v.box.map50)
        print(f'mAP@50: {hist["p1_map"]:.4f}')
    hist['p1_done'] = True
    save_hist()
else:
    print('Phase 1 done')

## Phase 2: 640px

In [ ]:
if not hist['p2_done']:
    print('\n=== PHASE 2: 640px (50 epochs) ===')
    best = CHECKPOINT_DIR / 'phase1_320px' / 'weights' / 'best.pt'
    m = YOLO(str(best))
    r = m.train(
        data=str(yaml_path), imgsz=640, epochs=50, batch=16,
        device=0 if torch.cuda.is_available() else -1,
        patience=30, save=True, save_period=10,
        project=str(CHECKPOINT_DIR), name='phase2_640px', exist_ok=True,
        lr0=0.0005, warmup_epochs=5, warmup_momentum=0.8, momentum=0.937, weight_decay=0.0005,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=20, translate=0.15, scale=0.5,
        flipud=0.5, fliplr=0.5, mosaic=1.0, val=True, verbose=True
    )
    best = Path(r.save_dir) / 'weights' / 'best.pt'
    if best.exists():
        v = YOLO(str(best)).val(data=str(yaml_path), split='val', verbose=False)
        hist['p2_map'] = float(v.box.map50)
        print(f'mAP@50: {hist["p2_map"]:.4f}')
    hist['p2_done'] = True
    save_hist()
else:
    print('Phase 2 done')

## Phase 3: 1024px (Optional)

In [ ]:
RUN_P3 = True
if RUN_P3 and not hist['p3_done']:
    print('\n=== PHASE 3: 1024px (30 epochs) ===')
    best = CHECKPOINT_DIR / 'phase2_640px' / 'weights' / 'best.pt'
    m = YOLO(str(best))
    r = m.train(
        data=str(yaml_path), imgsz=1024, epochs=30, batch=8,
        device=0 if torch.cuda.is_available() else -1,
        patience=15, save=True, save_period=5,
        project=str(CHECKPOINT_DIR), name='phase3_1024px', exist_ok=True,
        lr0=0.0001, warmup_epochs=2, momentum=0.937, weight_decay=0.0005,
        hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, degrees=10, translate=0.1, scale=0.3,
        flipud=0.3, fliplr=0.5, mosaic=0.8, val=True, verbose=True
    )
    best = Path(r.save_dir) / 'weights' / 'best.pt'
    if best.exists():
        v = YOLO(str(best)).val(data=str(yaml_path), split='val', verbose=False)
        hist['p3_map'] = float(v.box.map50)
        print(f'mAP@50: {hist["p3_map"]:.4f}')
    hist['p3_done'] = True
    save_hist()
elif hist['p3_done']:
    print('Phase 3 done')
else:
    print('Phase 3 disabled')

## Best Model & Threshold Tuning

In [ ]:
# Find best
best_phase = max(
    [(f'P1 (320px)', hist['p1_map'], CHECKPOINT_DIR / 'phase1_320px' / 'weights' / 'best.pt', hist['p1_done']),
     (f'P2 (640px)', hist['p2_map'], CHECKPOINT_DIR / 'phase2_640px' / 'weights' / 'best.pt', hist['p2_done']),
     (f'P3 (1024px)', hist['p3_map'], CHECKPOINT_DIR / 'phase3_1024px' / 'weights' / 'best.pt', hist['p3_done'])],
    key=lambda x: x[1] if x[3] else -1
)

name, score, path, _ = best_phase
print(f'Best: {name} (mAP@50={score:.4f})')

# Threshold sweep
print('\nTesting thresholds...')
m = YOLO(str(path))
best_conf = 0.5
best_map = 0
results = {}

for conf in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    m.conf = conf
    v = m.val(data=str(yaml_path), split='test', device=0 if torch.cuda.is_available() else -1, verbose=False)
    map50 = float(v.box.map50)
    results[conf] = {'map50': map50, 'precision': float(v.box.mp), 'recall': float(v.box.mr)}
    print(f'Conf {conf:.2f}: mAP@50={map50:.4f}')
    if map50 > best_map:
        best_map = map50
        best_conf = conf

print(f'\nOptimal: conf={best_conf:.2f} (mAP@50={best_map:.4f})')

# Save config
cfg = {
    'checkpoint': str(path),
    'confidence': float(best_conf),
    'map50': float(best_map),
    'precision': float(results[best_conf]['precision']),
    'recall': float(results[best_conf]['recall']),
    'thresholds': {float(k): v for k, v in results.items()},
    'model': 'YOLOv8l',
}
json.dump(cfg, open(CHECKPOINT_DIR / 'config.json', 'w'), indent=2)

print(f'\nConfig saved')
print(f'Next: Run 04_cross_dataset_eval.ipynb and 06_failure_analysis.ipynb')

## SAHI + Ensemble Evaluation

In [ ]:
# Ensemble Voting (Average Confidence across Phase 1, 2, 3)
print('\n=== ENSEMBLE EVALUATION ===')

p1_ckpt = CHECKPOINT_DIR / 'phase1_320px' / 'weights' / 'best.pt'
p2_ckpt = CHECKPOINT_DIR / 'phase2_640px' / 'weights' / 'best.pt'
p3_ckpt = CHECKPOINT_DIR / 'phase3_1024px' / 'weights' / 'best.pt'

test_img_dir = CHECKPOINT_DIR / 'test' / 'images'
test_images = sorted(list(test_img_dir.glob('*.*')))
print(f'Test images: {len(test_images)}')

# Collect predictions from all phases
from collections import defaultdict
all_preds = {}  # {img_stem: [conf_p1, conf_p2, conf_p3, ...]}

for ckpt_path in [p1_ckpt, p2_ckpt, p3_ckpt]:
    if not ckpt_path.exists():
        print(f'Skipping {ckpt_path.parent.name} (not found)')
        continue

    print(f'\nPredicting with {ckpt_path.parent.name}...')
    m = YOLO(str(ckpt_path))
    m.conf = best_conf

    for img_path in tqdm(test_images):
        r = m.predict(img_path, verbose=False)[0]
        img_stem = Path(img_path).stem
        if img_stem not in all_preds:
            all_preds[img_stem] = []

        if r.boxes is not None and len(r.boxes) > 0:
            confs = r.boxes.conf.cpu().numpy() if hasattr(r.boxes.conf, 'cpu') else r.boxes.conf
            all_preds[img_stem].append(float(confs.mean()) if len(confs) > 0 else 0.0)
        else:
            all_preds[img_stem].append(0.0)

# Average confidence across phases
ensemble_confs = {}
for img_stem, conf_list in all_preds.items():
    if conf_list:
        ensemble_confs[img_stem] = np.mean(conf_list)

avg_ensemble = np.mean(list(ensemble_confs.values())) if ensemble_confs else 0
print(f'\nEnsemble avg confidence: {avg_ensemble:.4f}')

# Validate each phase + ensemble on test split
print(f'\n--- Test Split Validation ---')
for phase, ckpt_path in [('P1 (320px)', p1_ckpt), ('P2 (640px)', p2_ckpt), ('P3 (1024px)', p3_ckpt)]:
    if ckpt_path.exists():
        m = YOLO(str(ckpt_path))
        v = m.val(data=str(yaml_path), split='test', device=0 if torch.cuda.is_available() else -1, verbose=False)
        print(f'{phase}: mAP@50 = {float(v.box.map50):.4f}')

print('\n✓ Ensemble evaluation complete')